In [1]:
!mkdir /kaggle/working/washed_audio

In [ ]:
import os
import gc
import numpy as np
import pandas as pd
import librosa
import soundfile as sf
from scipy.signal import butter, lfilter
from sklearn.linear_model import LinearRegression
from tqdm import tqdm
import multiprocessing as mp
from functools import partial

import tensorflow as tf
import tensorflow_hub as hub
import h5py
from tqdm import tqdm


import sys
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split


# Constants
LOWCUT = 500.0
HIGHCUT = 8000.0
TARGET_SR = 32000
N_FFT = 2048
HOP_LENGTH = 512
CHUNK_SIZE_SEC = 30 

def worker_init():
    """Environment safety for worker processes."""
    os.environ["OPENBLAS_NUM_THREADS"] = "1"
    os.environ["MKL_NUM_THREADS"] = "1"
    os.environ["OMP_NUM_THREADS"] = "1"
    os.environ["NUMBA_NUM_THREADS"] = "1"

def butter_bandpass(lowcut, highcut, fs, order=5):
    nyq = 0.5 * fs
    low = lowcut / nyq
    high = highcut / nyq
    b, a = butter(order, [low, high], btype='band')
    return b, a

def butter_bandpass_filter(data, lowcut, highcut, fs, order=5):
    b, a = butter_bandpass(lowcut, highcut, fs, order=order)
    y_filtered = lfilter(b, a, data)
    return y_filtered

def extract_features(y, sr, n_fft, hop_length):
    if len(y.shape) > 1:
        y = np.mean(y, axis=1)
    
    rms = librosa.feature.rms(y=y, frame_length=n_fft, hop_length=hop_length)[0]
    zcr = librosa.feature.zero_crossing_rate(y=y, frame_length=n_fft, hop_length=hop_length)[0]
    rolloff = librosa.feature.spectral_rolloff(y=y, sr=sr, n_fft=n_fft, hop_length=hop_length)[0]
    mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
    
    X_base = np.vstack((rms, zcr, rolloff)).T
    X_mfcc = mfccs.T
    
    min_f = min(X_base.shape[0], X_mfcc.shape[0])
    return np.hstack((X_base[:min_f], X_mfcc[:min_f]))

def train_vad_model(audio_dir):
    print("Training simple VAD model on a sample file...")
    sample_file = None
    for root, _, files in os.walk(audio_dir):
        for f in files:
            if f.endswith('.ogg') or f.endswith('.wav'):
                sample_file = os.path.join(root, f)
                break
        if sample_file: break
    
    if not sample_file:
        raise ValueError("No audio files found for training.")

    y, sr = librosa.load(sample_file, sr=TARGET_SR, duration=60)
    y_filtered = butter_bandpass_filter(y, LOWCUT, HIGHCUT, sr, order=6)
    features = extract_features(y_filtered, sr, N_FFT, HOP_LENGTH)
    
    rms = features[:, 0]
    zcr = features[:, 1]
    rolloff = features[:, 2]
    
    labels = ( (rms > np.mean(rms)*1.5) & (zcr > np.mean(zcr)*0.5) & (zcr < np.mean(zcr)*2.0) & 
               (rolloff > 1000) & (rolloff < 7000) ).astype(int)
    
    model = LinearRegression()
    model.fit(features, labels)
    print("VAD model trained successfully.")
    return model

def process_file_wrapper(input_path, output_dir, model):
    """Encapsulated processing for workers."""
    try:
        info = sf.info(input_path)
        orig_sr = info.samplerate
        block_size = CHUNK_SIZE_SEC * orig_sr
        
        all_y_filtered, all_features = [], []
        
        for block in sf.blocks(input_path, blocksize=block_size):
            y_chunk = np.mean(block, axis=1) if len(block.shape) > 1 else block
            if orig_sr != TARGET_SR:
                y_chunk = librosa.resample(y_chunk, orig_sr=orig_sr, target_sr=TARGET_SR)
            
            y_filt = butter_bandpass_filter(y_chunk, LOWCUT, HIGHCUT, TARGET_SR, order=6)
            all_y_filtered.append(y_filt)
            
            feats = extract_features(y_filt, TARGET_SR, N_FFT, HOP_LENGTH)
            all_features.append(feats)
            del block, y_chunk, y_filt, feats
            
        if not all_y_filtered: return input_path

        full_y_filt = np.concatenate(all_y_filtered)
        full_feats = np.vstack(all_features)
        
        vad_mask = (model.predict(full_feats) > 0.5).astype(bool)
        vad_samples = np.repeat(vad_mask, HOP_LENGTH)[:len(full_y_filt)]
        signal_audio = full_y_filt * vad_samples
        
        # out_path = os.path.join(output_dir, os.path.relpath(input_path, 'train_audio'))
        sub_path = input_path.split('train_audio/')[-1]
        out_path = os.path.join(output_dir, sub_path)


        
        os.makedirs(os.path.dirname(out_path), exist_ok=True)
        sf.write(out_path, signal_audio, TARGET_SR)
        
        del full_y_filt, full_feats, vad_mask, vad_samples, signal_audio, all_y_filtered, all_features
        return input_path
    except Exception as e:
        return f"ERROR:{input_path}:{str(e)}"




if __name__ == "__main__":
    import argparse
    parser = argparse.ArgumentParser()
    parser.add_argument("--limit", type=int, default=None)
    parser.add_argument("--workers", type=int, default=None)

    args = parser.parse_args([])
    args, unknown = parser.parse_known_args()

    audio_dir = '/kaggle/input/competitions/birdclef-2026/train_audio'
    output_dir = '/kaggle/working/washed_audio'
    
    if not os.path.exists(audio_dir):
        print(f"Directory {audio_dir} not found."); exit(1)
        
    model = train_vad_model(audio_dir)
    
    audio_files = []
    for root, _, files in os.walk(audio_dir):
        for f in files:
            if f.endswith('.ogg') or f.endswith('.wav'):
                audio_files.append(os.path.join(root, f))
    
    if args.limit:
        audio_files = audio_files[:args.limit]
    
    print(f"Starting parallel processing of {len(audio_files)} files with {args.workers} workers...")
    
    # Process files in parallel
    # maxtasksperchild=1 ensures a fresh process (and memory memory state) for every file
    with mp.Pool(processes=args.workers, initializer=worker_init, maxtasksperchild=1) as pool:
        process_func = partial(process_file_wrapper, output_dir=output_dir, model=model)
        for result in tqdm(pool.imap_unordered(process_func, audio_files), total=len(audio_files)):
            if isinstance(result, str) and result.startswith("ERROR"):
                print(result)

    print("Complete.")


2026-05-02 01:51:29.500728: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777686689.722425      57 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777686689.789944      57 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777686690.269198      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777686690.269244      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777686690.269247      57 computation_placer.cc:177] computation placer alr

Training simple VAD model on a sample file...
VAD model trained successfully.
Starting parallel processing of 35549 files with None workers...


  1%|          | 314/35549 [01:41<2:20:06,  4.19it/s]/usr/local/lib/python3.12/dist-packages/librosa/core/spectrum.py:266: UserWarning: n_fft=2048 is too large for input signal of length=1536
  warnings.warn(
  1%|          | 331/35549 [01:46<2:57:06,  3.31it/s]/usr/local/lib/python3.12/dist-packages/librosa/core/spectrum.py:266: UserWarning: n_fft=2048 is too large for input signal of length=586
  warnings.warn(
  1%|▏         | 462/35549 [02:20<2:12:17,  4.42it/s]/usr/local/lib/python3.12/dist-packages/librosa/core/spectrum.py:266: UserWarning: n_fft=2048 is too large for input signal of length=1536
  warnings.warn(
  2%|▏         | 545/35549 [02:40<2:08:27,  4.54it/s]/usr/local/lib/python3.12/dist-packages/librosa/core/spectrum.py:266: UserWarning: n_fft=2048 is too large for input signal of length=104
  warnings.warn(
  2%|▏         | 627/35549 [03:00<2:18:59,  4.19it/s]/usr/local/lib/python3.12/dist-packages/librosa/core/spectrum.py:266: UserWarning: n_fft=2048 is too large for in

In [ ]:


# Constants
TARGET_SR = 32000
SEGMENT_LEN = 5  # seconds
BATCH_SIZE = 1 # Keep it small for memory
MODEL_PATH = "/kaggle/input/models/google/bird-vocalization-classifier/tensorflow2/bird-vocalization-classifier/8"

def print_err(msg):
    sys.stderr.write(str(msg) + "\n")
    sys.stderr.flush()

def load_model():
    print_err("Loading Google BVC model...")
    model = tf.saved_model.load(MODEL_PATH)
    infer = model.signatures['serving_default']
    return infer

def extract_features(audio_path, infer):
    try:
        y, sr = sf.read(audio_path)
        if sr != TARGET_SR:
            # We should have resampled in preprocessing, but just in case
            import librosa
            y = librosa.resample(y, orig_sr=sr, target_sr=TARGET_SR)
        
        # Split into 5s segments
        num_samples = len(y)
        samples_per_seg = TARGET_SR * SEGMENT_LEN
        num_segments = num_samples // samples_per_seg
        
        embeddings = []
        for i in range(num_segments):
            start = i * samples_per_seg
            end = start + samples_per_seg
            chunk = y[start:end].astype(np.float32)
            
            # Ensure chunk is 1D and has correct length
            if len(chunk) < samples_per_seg:
                continue
            
            # Run inference
            out = infer(tf.constant(chunk[np.newaxis, :]))
            emb = out['embedding'].numpy()[0]
            embeddings.append(emb)
            
        return np.array(embeddings)
    except Exception as e:
        print_err(f"Error processing {audio_path}: {e}")
        return None

def main():
    import argparse
    parser = argparse.ArgumentParser()
    parser.add_argument("--limit", type=int, default=None)
    args = parser.parse_args()

    # Setup HDF5
    output_h5 = "embeddings.h5"
    if os.path.exists(output_h5):
        os.remove(output_h5)
        
    infer = load_model()
    
    # Collect all audio files
    audio_tasks = []
    
    # washed_audio (train_audio subset)
    for root, _, files in os.walk("washed_audio"):
        for f in files:
            if f.endswith('.ogg'):
                audio_tasks.append(os.path.join(root, f))
                
    # train_soundscapes
    for root, _, files in os.walk("train_soundscapes"):
        for f in files:
            if f.endswith('.ogg'):
                audio_tasks.append(os.path.join(root, f))
    
    print_err(f"Found {len(audio_tasks)} total audio files.")
    if args.limit:
        audio_tasks = audio_tasks[:args.limit]
        print_err(f"Limited to {len(audio_tasks)} files.")
    
    with h5py.File(output_h5, 'w') as h5f:
        emb_ds = h5f.create_dataset('embeddings', (0, 1280), maxshape=(None, 1280), dtype='float32', compression='gzip')
        meta_ds = h5f.create_dataset('metadata', (0,), maxshape=(None,), dtype=h5py.string_dtype(encoding='utf-8'))
        
        count = 0
        for i, fpath in enumerate(tqdm(audio_tasks)):
            print_err(f"Processing {fpath}...")
            embs = extract_features(fpath, infer)
            if embs is not None and len(embs) > 0:
                print_err(f"Extracted {len(embs)} segments.")
                n_new = len(embs)
                emb_ds.resize((count + n_new, 1280))
                emb_ds[count:count+n_new] = embs
                
                # Metadata: file_path, segment_index
                for j in range(n_new):
                    meta_str = f"{fpath}|{j}"
                    meta_ds.resize((count + j + 1,))
                    meta_ds[count+j] = meta_str
                
                count += n_new
            
            # Memory management
            if i % 100 == 0:
                gc.collect()
                tf.keras.backend.clear_session()

    print_err(f"Finished! Total segments extracted: {count}")

if __name__ == "__main__":
    main()


In [ ]:

# Constants
TARGET_SR = 32000
SEGMENT_LEN = 5  # seconds
BATCH_SIZE = 1 # Keep it small for memory

MODEL_PATH = "/kaggle/input/models/google/bird-vocalization-classifier/tensorflow2/bird-vocalization-classifier/8" 


def print_err(msg):
    sys.stderr.write(str(msg) + "\n")
    sys.stderr.flush()

def load_model():
    print_err("Loading Google BVC model...")
    model = tf.saved_model.load(MODEL_PATH)
    infer = model.signatures['serving_default']
    return infer

def extract_features(audio_path, infer):
    try:
        y, sr = sf.read(audio_path)
        if sr != TARGET_SR:
            # We should have resampled in preprocessing, but just in case
            import librosa
            y = librosa.resample(y, orig_sr=sr, target_sr=TARGET_SR)
        
        # Split into 5s segments
        num_samples = len(y)
        samples_per_seg = TARGET_SR * SEGMENT_LEN
        num_segments = num_samples // samples_per_seg
        
        embeddings = []
        for i in range(num_segments):
            start = i * samples_per_seg
            end = start + samples_per_seg
            chunk = y[start:end].astype(np.float32)
            
            # Ensure chunk is 1D and has correct length
            if len(chunk) < samples_per_seg:
                continue
            
            # Run inference
            out = infer(tf.constant(chunk[np.newaxis, :]))
            emb = out['embedding'].numpy()[0]
            embeddings.append(emb)
            
        return np.array(embeddings)
    except Exception as e:
        print_err(f"Error processing {audio_path}: {e}")
        return None

def main():
    import argparse
    parser = argparse.ArgumentParser()
    parser.add_argument("--limit", type=int, default=None)
    args = parser.parse_args()

    # Setup HDF5
    output_h5 = "/kaggle/working/embeddings.h5"
    if os.path.exists(output_h5):
        os.remove(output_h5)
        
    infer = load_model()
    
    # Collect all audio files
    audio_tasks = []
    
    # washed_audio (train_audio subset)
    for root, _, files in os.walk("/kaggle/working/washed_audio"):
        for f in files:
            if f.endswith('.ogg'):
                audio_tasks.append(os.path.join(root, f))
                
    # train_soundscapes
    for root, _, files in os.walk("train_soundscapes"):
        for f in files:
            if f.endswith('.ogg'):
                audio_tasks.append(os.path.join(root, f))
    
    print_err(f"Found {len(audio_tasks)} total audio files.")
    if args.limit:
        audio_tasks = audio_tasks[:args.limit]
        print_err(f"Limited to {len(audio_tasks)} files.")
    
    with h5py.File(output_h5, 'w') as h5f:
        emb_ds = h5f.create_dataset('embeddings', (0, 1280), maxshape=(None, 1280), dtype='float32', compression='gzip')
        meta_ds = h5f.create_dataset('metadata', (0,), maxshape=(None,), dtype=h5py.string_dtype(encoding='utf-8'))
        
        count = 0
        for i, fpath in enumerate(tqdm(audio_tasks)):
            print_err(f"Processing {fpath}...")
            embs = extract_features(fpath, infer)
            if embs is not None and len(embs) > 0:
                print_err(f"Extracted {len(embs)} segments.")
                n_new = len(embs)
                emb_ds.resize((count + n_new, 1280))
                emb_ds[count:count+n_new] = embs
                
                # Metadata: file_path, segment_index
                for j in range(n_new):
                    meta_str = f"{fpath}|{j}"
                    meta_ds.resize((count + j + 1,))
                    meta_ds[count+j] = meta_str
                
                count += n_new
            
            # Memory management
            if i % 100 == 0:
                gc.collect()
                tf.keras.backend.clear_session()

    print_err(f"Finished! Total segments extracted: {count}")

if __name__ == "__main__":
    main()


In [ ]:

# Constants
EMBEDDING_DIM = 1280
H5_PATH = "/kaggle/working/embeddings.h5"
TAXONOMY_PATH = "/kaggle/input/competitions/birdclef-2026/taxonomy.csv"
TRAIN_CSV = "/kaggle/input/competitions/birdclef-2026/train.csv"
SOUNDSCAPES_CSV = "/kaggle/input/competitions/birdclef-2026/train_soundscapes_labels.csv"

class BirdClassifier(nn.Module):
    def __init__(self, num_classes):
        super(BirdClassifier, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(EMBEDDING_DIM, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )
        
    def forward(self, x):
        return self.net(x)

def load_data():
    print("Loading metadata...")
    taxonomy = pd.read_csv(TAXONOMY_PATH)
    species_list = taxonomy['inat_taxon_id'].unique().astype(str)
    species_to_idx = {s: i for i, s in enumerate(species_list)}
    num_classes = len(species_list)
    
    # Map washed_audio files to inat_taxon_id
    train_df = pd.read_csv(TRAIN_CSV)
    # The filename in train.csv is like "1161364/iNat1216197.ogg"
    # The path in metadata is like "washed_audio/1161364/iNat1216197.ogg|0"
    file_to_species = {}
    for _, row in train_df.iterrows():
        fpath = os.path.join("washed_audio", row['filename'])
        file_to_species[fpath] = [str(row['inat_taxon_id'])]
        
    # Map soundscapes to inat_taxon_id
    # train_soundscapes_labels.csv: filename,start,end,primary_label (semicolon separated)
    soundscape_labels = pd.read_csv(SOUNDSCAPES_CSV)
    ss_map = {}
    for _, row in soundscape_labels.iterrows():
        # meta in h5 will be "train_soundscapes/filename|segment_index"
        # We need to map (filename, segment_index) to labels
        # segment_index is start // 5
        try:
            # Parse start time "00:00:05" to seconds
            h, m, s = map(int, row['start'].split(':'))
            seg_idx = (h * 3600 + m * 60 + s) // 5
            fpath = os.path.join("train_soundscapes", row['filename'])
            labels = str(row['primary_label']).split(';')
            ss_map[f"{fpath}|{seg_idx}"] = labels
        except:
            continue

    print("Loading embeddings from HDF5...")
    with h5py.File(H5_PATH, 'r') as f:
        embeddings = f['embeddings'][:]
        metadata = f['metadata'][:]
        
    X = []
    y = []
    
    for i, meta in enumerate(tqdm(metadata)):
        meta_str = meta.decode('utf-8')
        fpath, seg_idx = meta_str.split('|')
        
        # Determine labels
        labels = []
        if fpath.startswith("/kaggle/working/washed_audio/washed_audio"):
            labels = file_to_species.get(fpath, [])
        elif fpath.startswith("/kaggle/input/competitions/birdclef-2026/train_soundscapes"):
            labels = ss_map.get(meta_str, [])
            
        if not labels:
            continue
            
        target = np.zeros(num_classes, dtype=np.float32)
        for s in labels:
            if s in species_to_idx:
                target[species_to_idx[s]] = 1.0
        
        X.append(embeddings[i])
        y.append(target)
        
    return np.array(X), np.array(y), species_list

def train():
    X, y, species_list = load_data()
    X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.1, random_state=42)
    
    train_ds = torch.utils.data.TensorDataset(torch.tensor(X_train), torch.tensor(y_train))
    val_ds = torch.utils.data.TensorDataset(torch.tensor(X_val), torch.tensor(y_val))
    
    train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=64, shuffle=False)
    
    model = BirdClassifier(len(species_list))
    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.Adam(model.parameters(), lr=1e-3)
    
    epochs = 20
    for epoch in range(epochs):
        model.train()
        train_loss = 0
        for batch_x, batch_y in train_loader:
            optimizer.zero_grad()
            outputs = model(batch_x)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
            
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for batch_x, batch_y in val_loader:
                outputs = model(batch_x)
                loss = criterion(outputs, batch_y)
                val_loss += loss.item()
                
        print(f"Epoch {epoch+1}/{epochs}, Train Loss: {train_loss/len(train_loader):.4f}, Val Loss: {val_loss/len(val_loader):.4f}")
        
    torch.save({
        'model_state_dict': model.state_dict(),
        'species_list': species_list
    }, "/kaggle/working/bird_model.pth")
    print("Model and species list saved to bird_model.pth")

if __name__ == "__main__":
    train()
